# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL (Croissant schema)
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Access the metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.
This step explores the structure of the dataset using the `@id` fields that uniquely identify all entities in Croissant.

In [ ]:
# List all record sets by their @id and name
print("Available record sets in this dataset:")
record_sets_info = []
for record_set in dataset.metadata.record_sets:
    print(f"- @id: {record_set.id}")
    print(f"  name: {record_set.name}")
    print(f"  description: {record_set.description}")
    record_sets_info.append(record_set.id)
    # List field IDs for the record set
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - @id: {field.id}, name: {field.name}, data_type: {field.data_type}")
    print()

# For further steps, pick the first record set (most datasets have only one or a few)
if record_sets_info:
    default_record_set_id = record_sets_info[0]
else:
    raise ValueError('No record sets found in metadata. Check dataset definition.')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Always reference the record set by its `@id` as above.
We'll load **all record sets** here, but focus on one for demonstration.

In [ ]:
# Extract data from each available record set
dataframes = {}

for record_set in dataset.metadata.record_sets:
    print(f"Loading records from record set: {record_set.id} ({record_set.name})")
    records = list(dataset.records(record_set=record_set.id))
    df = pd.DataFrame(records)
    dataframes[record_set.id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Number of records: {len(df)}")

# For demonstration, pick the first record set
record_set_id = default_record_set_id

print(f"\nFirst 5 records from record set: {record_set_id}")
dataframes[record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps (e.g., filter, normalize, group, etc.)

*Below, update numeric_field and group_field with relevant `@id`s or column names as observed above for this dataset.*

In [ ]:
# Inspect DataFrame columns to choose a numeric field and group field
print(f"Available columns in {record_set_id}:")
print(dataframes[record_set_id].columns.tolist())

# For EDA, set these to appropriate column names (from a real run you will see e.g. 'cr:peptide_count', etc.)
# Examples below—adjust to actual names in your dataset
numeric_field = 'cr:Coverage'  # Replace with a numeric field @id from your dataset
threshold = 10 # example filter threshold

df = dataframes[record_set_id]
if numeric_field in df.columns:
    # Filtering
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold} (showing top 5):")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records (showing top 5):")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
else:
    print(f"Column {numeric_field} not found! Please update `numeric_field` to one of: {df.columns.tolist()}")

# Example of grouping by another field (replace with real group field @id)
group_field = 'cr:Gene_Symbol'  # Replace with a suitable categorical/ID field
if group_field in df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"\nGrouped mean of {numeric_field} by {group_field} (top 5):")
    print(grouped_df.head())
else:
    print(f"Column {group_field} not found! Please update `group_field`.")

## 5. Visualization
Visualize the distribution of a numeric field or explore relationships between columns. Adjust `numeric_field` and `group_field` as appropriate for your data.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of a numeric field
if numeric_field in df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

# Boxplot by grouping field
if (numeric_field in df.columns) and (group_field in df.columns):
    # Only plot if group_field is not too high cardinality
    if df[group_field].nunique() < 30:
        plt.figure(figsize=(12,6))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()


## 6. Conclusion
In this notebook, we demonstrated how to load and explore the FAIR^2 mass spectrometry dataset with `mlcroissant`. Key findings depend on the selected fields and analyses—for example, filtering by protein coverage, normalizing values, and visualizing the distribution of numerical protein features.

#### Next Steps:
- Try filtering using other fields (e.g., peptide counts, MW, pI, etc., using their `@id`s)
- Run additional visualizations or group-based summaries
- Save processed data subsets for downstream machine learning or statistical analysis
